# Ejercicio 5: Espacio Vectorial

## Objetivo de la práctica
- Implementar un Sistema de Recuperación de Información completo, desde la lectura del corpus hasta la recuperación de resultados.

## Parte 0: Carga del Corpus

Vamos a utilizar la API de Kaggle para acceder al dataset _Wikipedia Text Corpus for NLP and LLM Projects_

El corpus está disponible desde este [link](https://www.kaggle.com/datasets/gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects?utm_source=chatgpt.com)

### Actividad

1. Carga el corpus
2. Realiza las etapas de preprocesamiento sobre el corpus


In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\Erick\.cache\kagglehub\datasets\gzdekzlkaya\wikipedia-text-corpus-for-nlp-and-llm-projects\versions\1


In [4]:
import pandas as pd
corpus = pd.read_csv(path + "/wikipedia_text_corpus.csv")

In [5]:
print(corpus.columns)

Index(['Unnamed: 0', 'text'], dtype='str')


In [6]:
import re
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import nltk

# Descargar stopwords
nltk.download("stopwords")

stop_words = set(stopwords.words("english"))

# Stemmer
stemmer = PorterStemmer()

# Función de preprocesamiento
def preprocess_text(text):
    # Tokenización
    tokens = re.findall(r'\b[a-z]+\b', str(text).lower())

    # Eliminar stopwords
    tokens = [word for word in tokens if word not in stop_words]

    # Stemming
    tokens = [stemmer.stem(word) for word in tokens]

    return tokens

# Aplicar al DataFrame
corpus["tokens"] = corpus["text"].apply(preprocess_text)

# Mostrar teto
print(corpus[["text", "tokens"]].head())

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Erick\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


                                                text  \
0  Anovo\n\nAnovo (formerly A Novo) is a computer...   
1  Battery indicator\n\nA battery indicator (also...   
2  Bob Pease\n\nRobert Allen Pease (August 22, 19...   
3  CAVNET\n\nCAVNET was a secure military forum w...   
4  CLidar\n\nThe CLidar is a scientific instrumen...   

                                              tokens  
0  [anovo, anovo, formerli, novo, comput, servic,...  
1  [batteri, indic, batteri, indic, also, known, ...  
2  [bob, peas, robert, allen, peas, august, june,...  
3  [cavnet, cavnet, secur, militari, forum, becam...  
4  [clidar, clidar, scientif, instrument, use, me...  


## Parte 1: Recuperación con TF-IDF

### Actividad:
3. Obtén la representación vectorial de los documentos utilizando el modelo TF-IDF
4. A partir de un conjunto de 10 queries, verifica la recuperación del sistema

In [7]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

documents = corpus["text"].astype(str)

# =========================
# Vectorización TF-IDF
# =========================
vectorizer = TfidfVectorizer()

tfidf_matrix = vectorizer.fit_transform(documents)

print("Shape matriz de TF-IDF:")
print(tfidf_matrix.shape)

# Queries de prueba
queries = [
    "city government",
    "Digital Assets",
    "history of europe",
    "climate change effects",
    "space exploration nasa",
    "world war",
    "human brain function",
    "internet and technology",
    "renewable energy",
    "web hosting"
]


def retrieve_documents(query, top_k=3):

    # Vectorizar query
    query_vector = vectorizer.transform([query])

    # Similitud coseno
    similarities = cosine_similarity(query_vector, tfidf_matrix).flatten()

    # indices más relevantes
    top_indices = similarities.argsort()[-top_k:][::-1]

    print(f"\nQUERY: {query}\n")

    for rank, idx in enumerate(top_indices, start=1):
        print(f"Top {rank} | Score: {similarities[idx]:.4f}")
        print(documents.iloc[idx][:300])  # Mostrar primeros caracteres
        print("-" * 80)


# Evaluar las 10 queries
for query in queries:
    retrieve_documents(query)

Shape matriz de TF-IDF:
(10859, 148411)

QUERY: city government

Top 1 | Score: 0.3114
I-City

i-City is a 72 acre ICT-based urban development beside the Federal Highway in Section 7, Shah Alam, Selangor, Malaysia. Planned by architect Jon A. Jerde, i-City was designed as a fully integrated intelligent city, comprising corporate, leisure and residential components such as a 1 million 
--------------------------------------------------------------------------------
Top 2 | Score: 0.2490
3D city models

3D city models are digital models of urban areas that represent terrain surfaces, sites, buildings, vegetation, infrastructure and landscape elements in three-dimensional scale as well as related objects (e.g., city furniture) belonging to urban areas. Their components are described 
--------------------------------------------------------------------------------
Top 3 | Score: 0.2329
HITEC City

The Hyderabad Information Technology and Engineering Consultancy City, abbreviated as HITEC C

## Parte 2: Recuperación con BM25

### Actividad:
5. Implementa un sistema de recuperación usando el modelo BM25.
6. Para el mismo conjunto de 10 queries, verifica la recuperación del sistema

In [12]:
from rank_bm25 import BM25Okapi
import numpy as np

bm25 = BM25Okapi(corpus["tokens"])

for query in queries:
    scores = bm25.get_scores(preprocess_text(query))

    top_idx = np.argsort(scores)[::-1]  # orden descendente

    print("\nQUERY:", query)
    for rank, i in enumerate(top_idx[:3], start=1):
        print(f"Top {rank} | Score: {scores[i]:.4f}")
        print(documents.iloc[i][:300])
        print("-" * 80)


QUERY: city government
Top 1 | Score: 8.3589
HITEC City

The Hyderabad Information Technology and Engineering Consultancy City, abbreviated as HITEC City, is an Information Technology, Engineering, Health informatics, and Bioinformatics campus in the suburbs of Hyderabad, Ranga Reddy district, India. Located 15kmÂ² from the centre of Hyderabad
--------------------------------------------------------------------------------
Top 2 | Score: 8.0898
One Touch Make Ready

One Touch Make Ready (also known as One Touch, and often abbreviated as OTMR) is the various statutes and local ordinances passed by various local governments and utilities in the United States, which require the owners of utility poles to allow a single construction crew to ma
--------------------------------------------------------------------------------
Top 3 | Score: 8.0478
Vivek Kundra

Vivek Kundra (born October 9, 1974) is an American administrator who served as the first chief information officer of the United Sta

## Parte 3: Comparación de resultados

### Actividad:
7. Verifica cuáles documentos son recuperados (y en qué orden) por cada modelo de recuperación 

In [13]:
for query in queries:
    # TF-IDF
    query_vec = vectorizer.transform([query])
    sim = cosine_similarity(query_vec, tfidf_matrix).flatten()
    tfidf_top = sim.argsort()[-3:][::-1]

    # BM25
    bm25_scores = bm25.get_scores(preprocess_text(query))
    bm25_top = np.argsort(bm25_scores)[-3:][::-1]

    print(f"\nQUERY: {query}")
    print(f"TF-IDF:  {list(tfidf_top)} (scores: {sim[tfidf_top].round(4)})")
    print(f"BM25:    {list(bm25_top)} (scores: {bm25_scores[bm25_top].round(4)})")
    print(f"Iguales: {set(tfidf_top) == set(bm25_top)}")




QUERY: city government
TF-IDF:  [np.int64(3774), np.int64(3506), np.int64(1609)] (scores: [0.3114 0.249  0.2329])
BM25:    [np.int64(1609), np.int64(6140), np.int64(3304)] (scores: [8.3589 8.0898 8.0478])
Iguales: False

QUERY: Digital Assets
TF-IDF:  [np.int64(7752), np.int64(1047), np.int64(9928)] (scores: [0.4978 0.2931 0.2403])
BM25:    [np.int64(3963), np.int64(7386), np.int64(2877)] (scores: [11.2138 10.919  10.7236])
Iguales: False

QUERY: history of europe
TF-IDF:  [np.int64(6079), np.int64(1600), np.int64(3981)] (scores: [0.3309 0.3166 0.312 ])
BM25:    [np.int64(504), np.int64(3164), np.int64(3958)] (scores: [9.3379 7.9612 7.7899])
Iguales: False

QUERY: climate change effects
TF-IDF:  [np.int64(10805), np.int64(2547), np.int64(941)] (scores: [0.2984 0.2878 0.2377])
BM25:    [np.int64(954), np.int64(4062), np.int64(941)] (scores: [12.0982 10.8213 10.6504])
Iguales: False

QUERY: space exploration nasa
TF-IDF:  [np.int64(4083), np.int64(7573), np.int64(1066)] (scores: [0.5556